In [1]:
# we will be using the hugging face Math-Verify. This is a colloquial verifier used in many papers. 
!pip install "math-verify[antlr4_13_2]" 


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
def get_last_boxed(text):
    # Find the last \boxed or \fbox, whichever comes later
    idx_boxed = text.rfind(r"\boxed")
    idx_fbox = text.rfind(r"\fbox")
    if idx_boxed == -1 and idx_fbox == -1:
        return None
    if idx_boxed >= idx_fbox:
        current_idx = idx_boxed + len(r"\boxed")
    else:
        current_idx = idx_fbox + len(r"\fbox")

    # Skip whitespace
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # No brace: handle "\boxed 5" by taking the next token
    if current_idx >= len(text):
        return None
    if text[current_idx] != "{":
        token = text[current_idx:].split()[0].rstrip("$.")
        return token or None

    # Parse nested braces
    current_idx += 1
    brace_depth = 1
    content_start_idx = current_idx
    while current_idx < len(text) and brace_depth > 0:
        if text[current_idx] == "{":
            brace_depth += 1
        elif text[current_idx] == "}":
            brace_depth -= 1
        current_idx += 1

    if brace_depth != 0:
        return None

    answer = text[content_start_idx:current_idx - 1].strip()
    return answer or None

In [10]:
# the grader

from math_verify import parse, verify

def mv_grade(gold_answer, response):
    # True if last \boxed{} in response matches gold_answer (Math-Verify)
    boxed = get_last_boxed(response)
    if boxed is None:
        return False  # no box = no credit (formatting rule)
    try:
        gold = parse(f"${gold_answer}$")
        pred = parse(f"${boxed}$")
        return verify(gold, pred)   # gold goes first
    except Exception:
        return False  # a parse failure or timeout counts as wrong


In [ ]:
# test suite

cases = [  # (gold, model response, expected)
    (r"\frac{1}{2}", r"so $\boxed{0.5}$", True),
    (r"\frac{3}{4}", r"\boxed{\frac34}", True),
    (r"\frac{\sqrt{2}}{2}", r"\boxed{\sqrt{\frac{1}{2}}}", True),
    (r"2\pi", r"\boxed{2 \pi}", True),
    (r"\pi", r"\boxed{3}", False),
    (r"5\sqrt{2}", r"\boxed{\sqrt{50}}", True),
    (r"120^\circ", r"\boxed{120}", True),
    (r"5", r"\boxed{x=5}", True),
    (r"10\%", r"\boxed{10}", True),
    (r"5", r"\boxed{5 \text{ cm}}", True),
    (r"(x+1)^2", r"\boxed{x^2+2x+1}", True),
    (r"(0,1)", r"\boxed{[0,1]}", False),
    (r"\{1,2,3\}", r"\boxed{\{3,2,1\}}", True),
    (r"-2, 3", r"\boxed{3, -2}", True),
    (r"\begin{pmatrix} 1 \\ 2 \end{pmatrix}", r"\boxed{\begin{pmatrix} 2 \\ 1 \end{pmatrix}}", False),
    (r"\frac{7}{2}", r"\boxed{3}", False),
    (r"3", r"the answer is 3", False),      # no box
    (r"3", r"\boxed{\frac{3", False),       # truncated output
]
fails = 0
for gold, resp, expected in cases:
    got = mv_grade(gold, resp)
    fails += got != expected
    print(f"{'OK  ' if got == expected else 'FAIL'} {gold!r:40} | {resp!r:42} -> {got}")
assert fails == 0, f"{fails} grader tests failed"

OK   '\\frac{1}{2}'                           | 'so $\\boxed{0.5}$'                        -> True
OK   '\\frac{3}{4}'                           | '\\boxed{\\frac34}'                        -> True
OK   '\\frac{\\sqrt{2}}{2}'                   | '\\boxed{\\sqrt{\\frac{1}{2}}}'            -> True
OK   '2\\pi'                                  | '\\boxed{2 \\pi}'                          -> True
OK   '\\pi'                                   | '\\boxed{3}'                               -> False
OK   '5\\sqrt{2}'                             | '\\boxed{\\sqrt{50}}'                      -> True
OK   '120^\\circ'                             | '\\boxed{120}'                             -> True
OK   '5'                                      | '\\boxed{x=5}'                             -> True
OK   '10\\%'                                  | '\\boxed{10}'                              -> True
OK   '5'                                      | '\\boxed{5 \\text{ cm}}'                   -> True
OK   '(x+

In [17]:
# we can also do a sanity check on MATH-500 where every gold_answer, boxed should grade as correct against itself

from datasets import load_from_disk
math500 = load_from_disk("data/math500")

bad = [ex for ex in math500 if not mv_grade(ex["answer"], f"\\boxed{{{ex['answer']}}}")]
print(f"{len(bad)} / {len(math500)} gold answers fail self-check")
for ex in bad[:10]:
    print(repr(ex["answer"]))

0 / 500 gold answers fail self-check


In [16]:
import sys; sys.path.insert(0, ".")
from posttrain import extract, grader
extract.run_tests(verbose=False); grader.run_tests(verbose=False)
print("posttrain OK")

posttrain OK


In [1]:
from posttrain.data import load_split
sft_all = load_split("sft_all")
math_val = load_split("math_val")
math500 = load_split("math500")
print(len(sft_all), len(math_val), len(math500))   # 122416, 502, 500

ModuleNotFoundError: No module named 'posttrain.data'